# Laguna State Polytechnic University
## College of Computer Studies
### CSST 102 – Basic Machine Learning | Final Project
---
# AniTaya: Agricultural Harvest and Income Prediction System
**A Web-Based Machine Learning System Predicting Crop Yield and Farmer Income in the Philippines**

**Dataset Citation:** Zenodo Record `18761663` (DOI: `10.5281/zenodo.18761663`)

**Target Variable:** `value_hg_ha` (Crop yield in hectograms per hectare)
**Primary Model:** Random Forest Regressor (R² = 0.9660)
**Baseline & Comparative Models:** Linear Regression, Ridge Regression, Decision Tree, Gradient Boosting

## 1. Import Libraries and Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import joblib

# Load dataset
data_path = os.path.join("..", "data", "df_yield_merged.csv")
df = pd.read_csv(data_path)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## 2. Exploratory Data Analysis (EDA) & Data Integrity Checks

In [ ]:
# Missing values check
print("=== MISSING VALUES ===")
print(df.isnull().sum())

# Data types and non-null summary
print("\n=== DATA TYPES ===")
print(df.dtypes)

# Crop distribution
print("\n=== CROP COUNTS ===")
print(df['item'].value_counts())

# Summary statistics
df.describe()

## 3. Preprocessing Pipeline & Data Splitting (Preventing Data Leakage)

In [ ]:
# Define Features (X) and Target (y)
feature_cols = ["item", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "area"]
target_col = "value_hg_ha"

X = df[feature_cols]
y = df[target_col]

# 80/20 Train-Test Split with fixed random_state for full reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

categorical_features = ["item", "area"]
numeric_features = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]

# ColumnTransformer preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

## 4. Model Training & Comparison
As required by CSST 102 guidelines, we benchmark the primary model against multiple baselines:
1. **Linear Regression** (Standard parametric baseline)
2. **Ridge Regression** (L2-regularized linear baseline)
3. **Decision Tree Regressor** (Single non-linear tree)
4. **Random Forest Regressor** (Primary Ensemble Model)
5. **Gradient Boosting Regressor** (Sequential boosting alternative)

In [ ]:
models = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=15, random_state=42),
    "Random Forest (Primary Model)": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = {}
trained_pipelines = {}
cv = KFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    cv_scores = cross_val_score(pipeline, X_train.iloc[:5000], y_train.iloc[:5000], cv=cv, scoring="r2", n_jobs=1)
    
    results[name] = {
        "MAE (hg/ha)": round(mae, 2),
        "RMSE (hg/ha)": round(rmse, 2),
        "R2 Score": round(r2, 4),
        "3-Fold CV R2": f"{cv_scores.mean():.4f} (±{cv_scores.std():.3f})"
    }
    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results).T
results_df

## 5. Model Evaluation & Visualization

In [ ]:
# Plot R2 scores
plt.figure(figsize=(9, 4), dpi=120)
bars = plt.barh(results_df.index, results_df['R2 Score'].astype(float), color=['#94a3b8', '#94a3b8', '#64748b', '#059669', '#0d9488'])
plt.xlim(0, 1.05)
plt.xlabel('R² Score (Higher is better)')
plt.title('CSST 102: Candidate Model Performance Comparison')
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature Importance Interpretation

In [ ]:
best_pipeline = trained_pipelines["Random Forest (Primary Model)"]
rf_model = best_pipeline.named_steps["regressor"]

cat_encoder = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_names = numeric_features + cat_names
importances = rf_model.feature_importances_

agg = {
    "Crop Type (item)": 0.0,
    "Country / Region (area)": 0.0,
    "Rainfall (mm/yr)": 0.0,
    "Pesticides/Agrochemicals": 0.0,
    "Average Temperature": 0.0
}
for name, val in zip(all_names, importances):
    if name.startswith("item_"):
        agg["Crop Type (item)"] += val
    elif name.startswith("area_"):
        agg["Country / Region (area)"] += val
    elif name == "average_rain_fall_mm_per_year":
        agg["Rainfall (mm/yr)"] += val
    elif name == "pesticides_tonnes":
        agg["Pesticides/Agrochemicals"] += val
    elif name == "avg_temp":
        agg["Average Temperature"] += val

fi_df = pd.DataFrame(list(agg.items()), columns=["Feature", "Importance"]).sort_values(by="Importance", ascending=True)
fi_df["Importance_Pct"] = fi_df["Importance"] * 100

plt.figure(figsize=(8, 4), dpi=120)
plt.barh(fi_df["Feature"], fi_df["Importance_Pct"], color="#10b981")
plt.xlabel("Importance (%)")
plt.title("AniTaya: Feature Importance Distribution")
plt.tight_layout()
plt.show()

## 7. Sample Prediction for Filipino Farmers

In [ ]:
# Example: 2.0 Hectares of Palay in Laguna during Wet Season
sample_input = pd.DataFrame([{
    "item": "Rice, paddy",
    "average_rain_fall_mm_per_year": 2400.0,
    "pesticides_tonnes": 120.0,
    "avg_temp": 28.2,
    "area": "Indonesia"
}])

predicted_hg_ha = best_pipeline.predict(sample_input)[0]
yield_mt_ha = predicted_hg_ha / 10000.0
land_area = 2.0 # hectares

total_yield_kg = (predicted_hg_ha / 10.0) * land_area
total_yield_mt = total_yield_kg / 1000.0
total_cavans = total_yield_kg / 50.0 # 50 kg per sack

farmgate_price_per_kg = 22.0 # PHP (DA benchmark)
cost_per_ha = 45000.0 # PHP

gross_revenue = total_yield_kg * farmgate_price_per_kg
total_cost = cost_per_ha * land_area
net_profit = gross_revenue - total_cost
roi = (net_profit / total_cost) * 100.0

print(f"Predicted Yield Rate: {yield_mt_ha:.2f} MT/ha")
print(f"Total Harvest Yield: {total_yield_mt:.2f} Metric Tons ({total_yield_kg:,.1f} kg)")
print(f"Total Cavans (50kg sacks): {total_cavans:.1f} cavans of palay")
print(f"Gross Revenue: PHP {gross_revenue:,.2f}")
print(f"Total Production Cost: PHP {total_cost:,.2f}")
print(f"Net Farmer Profit: PHP {net_profit:,.2f} (ROI: {roi:.1f}%)")